In [23]:
from cellm import ask
from ddgs import DDGS
import httpx, trafilatura

In [3]:
results = DDGS().text("Paris weather September 19 2023", max_results=5)

In [6]:
results[2]

{'title': 'Paris September 2023 Historical Weather Data (France) - Weather Spark',
 'href': 'https://weatherspark.com/h/m/47913/2023/9/Historical-Weather-in-September-2023-in-Paris-France',
 'body': 'See all nearby weather stations This report shows the past weather for Paris, providing a weather history for September 2023. It features all historical weather data series we have available, including the Paris temperature history for September 2023. You can drill down from year to month and even day level reports by clicking on the graphs.'}

In [20]:
url = results[2]["href"]
r = httpx.get(url, timeout=10, follow_redirects=True)
text = trafilatura.extract(r.text)

In [21]:
text

'| September 2023 Weather History in Paris France The data for this report comes from the Paris Orly Airport. See all nearby weather stations This report shows the past weather for Paris, providing a weather history for September 2023. It features all historical weather data series we have available, including the Paris temperature history for September 2023. You can drill down from year to month and even day level reports by clicking on the graphs. Hourly Wind Speed in September 2023 in Paris     0 mph   calm   1 mph   light air   4 mph   light breeze   8 mph   gentle breeze   13 mph   moderate breeze   18 mph   fresh breeze   25 mph   strong breeze   31 mph   near gale   39 mph   gale   47 mph   strong gale   55 mph   storm   64 mph   violent storm   73 mph   hurricane force   Growing degree days are a measure of yearly heat accumulation used to predict plant and animal development, and defined as the integral of warmth above a base temperature, discarding any excess above a maximum 

In [24]:
ask("What was the weather in Paris in September 2023?")

##### 🤖 LLM Response (glm-5.3-flash | in: 2892 | out: 620)

> Based on the extracted Weatherspark data (which lists observation days only, mostly rainy ones):
> 
> - **Rain days**: Sep 1, 11, 21, 29 (light rain); Sep 12 (heavy rain showers + mist/fog); Sep 22 (showers + mist)
> - **Fog/Mist**: Sep 13, 14, 24
> - **Thunderstorm nearby**: Sep 17
> 
> Note: **Sep 19 has no recorded weather observations** in this table — the page only lists days with notable events, so the remaining days were presumably clear/dry.
> 
> 💡 **Tip**: This page has no temperatures. For actual temp data, the [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api) is free and returns clean JSON — much easier than scraping:
> 
> ```python
> httpx.get("https://archive-api.open-meteo.com/v1/archive",
>           params={"latitude": 48.85, "longitude": 2.35,
>                   "start_date": "2023-09-01", "end_date": "2023-09-30",
>                   "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum"}).json()
> ```

```
tools = [
  {
    "name": "web_search",
    "description": (
      "Search the web and return a list of results (title, URL, short snippet). "
      "Use for finding pages about a topic, error message, library, or API. "
      "Returns snippets only, not full pages; use web_fetch to read a page. "
      "Keep queries short and specific (2-6 words). For code, add the library name "
      "or error text. Try a differently worded query if results are off-target."
    ),
    "input_schema": {
      "type": "object",
      "properties": {
        "query": {"type": "string", "description": "Search query, 2-6 words"}
      },
      "required": ["query"],
    },
  },
  {
    "name": "web_fetch",
    "description": (
      "Download one web page and return its text content (HTML stripped). "
      "Use after web_search when a snippet is not enough, or when the user gives a URL. "
      "Only pass URLs from search results or the user. "
      "Fetch the 1-3 most promising pages, not all of them. "
      "May fail on blocked or empty pages; then try the next result."
    ),
    "input_schema": {
      "type": "object",
      "properties": {
        "url": {"type": "string", "description": "Full URL including https://"}
      },
      "required": ["url"],
    },
  },
]
```